# 🫀 nnFormer: 3D Medical Image Segmentation with Swin-Transformers
### End-to-End Runnable Kaggle Notebook

This notebook provides a complete pipeline to test and train **nnFormer** on Kaggle GPUs (P100 / T4 / A100 / TPU).

#### Features Included:
1. **Environment Setup**: Directory structure, paths, and package installation.
2. **Dataset Generation**: Automated creation of 3D NIfTI cardiac MRI dataset (Task001_ACDC).
3. **Preprocessing**: Intensity normalization, cropping, and dataset planning (`nnFormer_plan_and_preprocess`).
4. **Model Training**: 3D Swin-Transformer network training (`nnFormer_train`).
5. **Inference**: Automated test volume prediction.
6. **Evaluation & Visualization**: Dice Similarity Coefficient (DSC) metrics calculation and 2D/3D slice visualization.

> 💡 **Tip:** Enable GPU accelerator in Kaggle Notebook settings (**Settings -> Accelerator -> GPU T4 x2 or P100**).

In [ ]:
# 1. Environment Configuration
import os
import sys
import torch

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))

# Configure nnFormer Environment Paths & PYTHONPATH
BASE_DIR = '/kaggle/working'
PROJECT_DIR = os.path.join(BASE_DIR, 'nextgen-transformer')
DATASET_DIR = os.path.join(BASE_DIR, 'DATASET')

os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['PYTHONPATH'] = PROJECT_DIR + ':' + os.environ.get('PYTHONPATH', '')
os.environ['nnFormer_raw_data_base'] = os.path.join(DATASET_DIR, 'nnFormer_raw')
os.environ['nnFormer_preprocessed'] = os.path.join(DATASET_DIR, 'nnFormer_preprocessed')
os.environ['RESULTS_FOLDER'] = os.path.join(DATASET_DIR, 'nnFormer_trained_models')
os.environ['nnFormer_def_n_proc'] = '2'
os.environ['nnFormer_n_proc_DA'] = '1'
os.environ['MAX_NUM_EPOCHS'] = '100'
os.environ['NUM_BATCHES_PER_EPOCH'] = '10'
os.environ['NUM_VAL_BATCHES_PER_EPOCH'] = '5'

print('\n--- Configured Paths ---')
print('PYTHONPATH:', os.environ['PYTHONPATH'])
print('nnFormer_raw_data_base:', os.environ['nnFormer_raw_data_base'])
print('nnFormer_preprocessed:', os.environ['nnFormer_preprocessed'])
print('RESULTS_FOLDER:', os.environ['RESULTS_FOLDER'])

In [ ]:
# 2. Clone Fresh Repository & Install Dependencies
%cd /kaggle/working
import sys
import os

!rm -rf /kaggle/working/nextgen-transformer
!git clone https://github.com/sharadpawarsaini/nextgen-transformer.git /kaggle/working/nextgen-transformer
%cd /kaggle/working/nextgen-transformer

!pip install --upgrade --force-reinstall -q batchgenerators==0.23 SimpleITK nibabel timm einops scipy matplotlib scikit-learn reportlab medpy
!pip install -e . --no-build-isolation

# Add repository to sys.path in live Jupyter kernel
PROJECT_DIR = '/kaggle/working/nextgen-transformer'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Dynamic runtime patch for batchgenerators namespace
import batchgenerators.dataloading
if not hasattr(batchgenerators.dataloading, 'MultiThreadedAugmenter'):
    try:
        from batchgenerators.dataloading.multi_threaded_augmenter import MultiThreadedAugmenter
        batchgenerators.dataloading.MultiThreadedAugmenter = MultiThreadedAugmenter
    except Exception:
        pass

print('✅ Repository cloned and sys.path configured successfully!')

In [ ]:
# 3. Dataset Generation (Task001_ACDC Synthetic / Demo Data)
import json
import numpy as np
import nibabel as nib

RAW_DATA = os.path.join(os.environ['nnFormer_raw_data_base'], 'nnFormer_raw_data')
TASK_ID = 1
TASK_NAME = f'Task{TASK_ID:03d}_ACDC'
TASK_DIR = os.path.join(RAW_DATA, TASK_NAME)

def create_sample_nifti(image_path, label_path, shape=(14, 160, 160)):
    D, H, W = shape
    img = np.random.normal(100.0, 15.0, shape).astype(np.float32)
    lbl = np.zeros(shape, dtype=np.uint8)
    zz, yy, xx = np.ogrid[:D, :H, :W]
    cz, cy, cx = D // 2, H // 2, W // 2

    # RV: Right Ventricle (Label 1)
    mask_rv = ((zz - cz)**2 / 4.0 + (yy - (cy - 15))**2 / 16.0 + (xx - (cx - 20))**2 / 16.0) <= 15.0
    lbl[mask_rv] = 1
    img[mask_rv] += 250.0

    # Myocardium (Label 2)
    mask_myo_outer = ((zz - cz)**2 / 5.0 + (yy - cy)**2 / 28.0 + (xx - cx)**2 / 28.0) <= 22.0
    mask_myo_inner = ((zz - cz)**2 / 5.0 + (yy - cy)**2 / 12.0 + (xx - cx)**2 / 12.0) <= 10.0
    mask_myo = mask_myo_outer & (~mask_myo_inner)
    lbl[mask_myo] = 2
    img[mask_myo] += 180.0

    # LV: Left Ventricle (Label 3)
    mask_lv = mask_myo_inner
    lbl[mask_lv] = 3
    img[mask_lv] += 320.0

    affine = np.eye(4)
    affine[0, 0] = 1.4
    affine[1, 1] = 1.4
    affine[2, 2] = 8.0

    nib_img = nib.Nifti1Image(img, affine)
    nib.save(nib_img, image_path)

    if label_path:
        nib_lbl = nib.Nifti1Image(lbl, affine)
        nib.save(nib_lbl, label_path)

# Prepare Directory Structure
for subfolder in ['imagesTr', 'labelsTr', 'imagesTs', 'labelsTs']:
    os.makedirs(os.path.join(TASK_DIR, subfolder), exist_ok=True)

print('Generating sample NIfTI 3D Cardiac MRI volumes...')
create_sample_nifti(
    os.path.join(TASK_DIR, 'imagesTr', 'patient001_0000.nii.gz'),
    os.path.join(TASK_DIR, 'labelsTr', 'patient001.nii.gz')
)
create_sample_nifti(
    os.path.join(TASK_DIR, 'imagesTr', 'patient002_0000.nii.gz'),
    os.path.join(TASK_DIR, 'labelsTr', 'patient002.nii.gz')
)
create_sample_nifti(
    os.path.join(TASK_DIR, 'imagesTs', 'patient003_0000.nii.gz'),
    os.path.join(TASK_DIR, 'labelsTs', 'patient003.nii.gz')
)

dataset_json = {
    'name': 'ACDC',
    'description': 'Automatic Cardiac Diagnosis Challenge Demo',
    'tensorImageSize': '4D',
    'reference': 'https://www.creatis.insa-lyon.fr/Challenge/acdc/',
    'licence': 'see ACDC website',
    'release': '1.0',
    'modality': {'0': 'MRI'},
    'labels': {'0': 'background', '1': 'RV', '2': 'Myo', '3': 'LV'},
    'numTraining': 2,
    'numTest': 1,
    'training': [
        {'image': './imagesTr/patient001.nii.gz', 'label': './labelsTr/patient001.nii.gz'},
        {'image': './imagesTr/patient002.nii.gz', 'label': './labelsTr/patient002.nii.gz'}
    ],
    'test': ['./imagesTs/patient003.nii.gz']
}

with open(os.path.join(TASK_DIR, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=4)

print(f'Task001_ACDC dataset setup complete in {TASK_DIR}')

In [ ]:
# 4. Experiment Planning & Data Preprocessing
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'
if not os.path.exists(os.path.join(PROJECT_DIR, 'nnformer')):
    print('Repository missing... Cloning automatically now!')
    subprocess.run(['git', 'clone', 'https://github.com/sharadpawarsaini/nextgen-transformer.git', PROJECT_DIR], cwd='/kaggle/working', check=True)

print('Starting nnFormer planning and preprocessing...')
cmd = [
    sys.executable,
    '-m', 'nnformer.experiment_planning.nnFormer_plan_and_preprocess',
    '-t', '1',
    '-tl', '2',
    '-tf', '2',
    '--verify_dataset_integrity'
]
subprocess.run(cmd, env=os.environ, cwd=PROJECT_DIR, check=True)
print('Preprocessing finished successfully!')

In [ ]:
# 5. Train nnFormer 3D Swin-Transformer Model
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'
if not os.path.exists(os.path.join(PROJECT_DIR, 'nnformer')):
    subprocess.run(['git', 'clone', 'https://github.com/sharadpawarsaini/nextgen-transformer.git', PROJECT_DIR], cwd='/kaggle/working', check=True)

print('Starting nnFormer model training...')
cmd = [
    sys.executable,
    '-m', 'nnformer.run.run_training',
    '3d_fullres',
    'nnFormerTrainerV2_nnformer_acdc',
    '1',
    '0'
]
subprocess.run(cmd, env=os.environ, cwd=PROJECT_DIR, check=True)
print('nnFormer model training completed!')

In [ ]:
# 6. Perform Inference on Test Set
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'
if not os.path.exists(os.path.join(PROJECT_DIR, 'nnformer')):
    subprocess.run(['git', 'clone', 'https://github.com/sharadpawarsaini/nextgen-transformer.git', PROJECT_DIR], cwd='/kaggle/working', check=True)

input_folder = os.path.join(TASK_DIR, 'imagesTs')
output_folder = os.path.join(TASK_DIR, 'inferTs', 'nnformer_acdc')
os.makedirs(output_folder, exist_ok=True)

cmd = [
    sys.executable,
    '-m', 'nnformer.inference.predict_simple',
    '-i', input_folder,
    '-o', output_folder,
    '-m', '3d_fullres',
    '-t', '1',
    '-tr', 'nnFormerTrainerV2_nnformer_acdc',
    '-chk', 'model_final_checkpoint'
]
subprocess.run(cmd, env=os.environ, cwd=PROJECT_DIR, check=True)
print(f'Inference complete! Predictions saved to: {output_folder}')

In [ ]:
# 7. Calculate Dice Similarity Coefficient (DSC) Metrics
def calculate_dice(pred, gt, class_id):
    pred_c = (pred == class_id)
    gt_c = (gt == class_id)
    intersection = np.logical_and(pred_c, gt_c).sum()
    denominator = pred_c.sum() + gt_c.sum()
    if denominator == 0:
        return 1.0
    return (2.0 * intersection) / denominator

pred_path = os.path.join(output_folder, 'patient003.nii.gz')
gt_path = os.path.join(TASK_DIR, 'labelsTs', 'patient003.nii.gz')

if os.path.exists(pred_path) and os.path.exists(gt_path):
    pred_img = nib.load(pred_path).get_fdata()
    gt_img = nib.load(gt_path).get_fdata()

    dsc_rv = calculate_dice(pred_img, gt_img, 1) * 100
    dsc_myo = calculate_dice(pred_img, gt_img, 2) * 100
    dsc_lv = calculate_dice(pred_img, gt_img, 3) * 100
    mean_dsc = (dsc_rv + dsc_myo + dsc_lv) / 3.0

    print('=' * 50)
    print('       nnFormer Segmentation Evaluation Results')
    print('=' * 50)
    print(f'  Right Ventricle (RV) DSC : {dsc_rv:.2f}%')
    print(f'  Myocardium (Myo) DSC     : {dsc_myo:.2f}%')
    print(f'  Left Ventricle (LV) DSC  : {dsc_lv:.2f}%')
    print('--------------------------------------------------')
    print(f'  Mean Dice Score (Accuracy): {mean_dsc:.2f}%')
    print('=' * 50)
else:
    print('Prediction files not found for evaluation.')

In [ ]:
# 8. Visualizing Results (Mid-Slice MRI vs Ground Truth vs Prediction)
import matplotlib.pyplot as plt

img_raw = nib.load(os.path.join(TASK_DIR, 'imagesTs', 'patient003_0000.nii.gz')).get_fdata()
gt_raw = nib.load(os.path.join(TASK_DIR, 'labelsTs', 'patient003.nii.gz')).get_fdata()
pred_raw = nib.load(os.path.join(output_folder, 'patient003.nii.gz')).get_fdata()

slice_idx = img_raw.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_raw[slice_idx], cmap='gray')
axes[0].set_title('Input 3D MRI Slice')
axes[0].axis('off')

axes[1].imshow(gt_raw[slice_idx], cmap='jet')
axes[1].set_title('Ground Truth Segmentation')
axes[1].axis('off')

axes[2].imshow(pred_raw[slice_idx], cmap='jet')
axes[2].set_title('nnFormer Prediction')
axes[2].axis('off')

plt.tight_layout()
plt.show()